In [1]:
import scanpy as sc
import scvi
import numpy as np
import sys
sys.path.append('../')

from scripts.subset_hvg import subset_to_hvg

adata = sc.read_h5ad(
    "../../data/obesity_challenge_2.h5ad"
)

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('../')
from scripts.pairing import assign_state_label
adata.obs["cell_state"] = assign_state_label(
    adata.obs
)
print(adata.obs["cell_state"].value_counts())

cell_state
other         43319
pre_adipo     26681
adipo         16057
lipo_adipo     4758
Name: count, dtype: int64


In [3]:
import pandas as pd

val_df = pd.read_csv(
    "../../data/preprocessed/val_perts/validation_mapping.csv"
)

val_condition = val_df["val_pert"].unique().tolist()

print(len(val_condition))
print(val_condition[:10])

34
['CEBPD+CREB1', 'KIF11+MLXIPL', 'CEBPD+SREBF1', 'CEBPD+ZBED3', 'CEBPB+PPARG', 'CEBPB+FOXO1', 'CEBPA+MLXIPL', 'STAT5A+TCF7L2', 'MLXIPL+STAT5A', 'CEBPB+NC']


In [4]:
adata_train = adata[
    ~adata.obs["gene"].isin(val_condition)
].copy()

adata_test = adata[
    adata.obs["gene"].isin(val_condition)
].copy()

In [5]:
adata_train_subset, hvg_genes,sig_genes = subset_to_hvg(
    adata_train,
    hvg_path="../../data/preprocessed/HVG/pertubation_genes.txt",
    include_signature_genes=True
)

adata_test_subset, hvg_genes,sig_genes = subset_to_hvg(
    adata_test,
    hvg_path="../../data/preprocessed/HVG/pertubation_genes.txt",
    include_signature_genes=True
)

HVG requested: 1939
Signature genes requested: 820
Signature genes used: 607
Total genes used: 2546
HVG requested: 1939
Signature genes requested: 820
Signature genes used: 607
Total genes used: 2546


In [6]:
# load model
scanvi = scvi.model.SCVI.load(
    "../../resources/scvi_model_pert_50",
    adata=adata_train_subset
)

# mapping
scvi.model.SCVI.prepare_query_anndata(
    adata_train_subset,
    scanvi
)

# latent
adata_train.obsm["X_scanvi"] = (
    scanvi.get_latent_representation(
        adata_train_subset
    )
)

print("latent shape:")
print(adata_train.obsm["X_scanvi"].shape)

import gc

# 刪除不再需要的物件
del adata_train_subset, adata

# 強制 python 釋放記憶體
gc.collect()

INFO     File ../../resources/scvi_model_pert_50/model.pt already downloaded                                       


/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/model/base/_base_model.py:862: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


INFO     Found 100.0% reference vars in query data.                                                                
latent shape:
(86693, 50)


593

In [7]:
adata_train.obsm["X_scanvi"].mean(), adata_train.obsm["X_scanvi"].std(),

(np.float32(-0.0004521801), np.float32(0.7905078))

In [8]:
# load model
scanvi = scvi.model.SCVI.load(
    "../../resources/scvi_model_pert_50",
    adata=adata_test_subset
)

# mapping
scvi.model.SCVI.prepare_query_anndata(
    adata_test_subset,
    scanvi
)

# latent
adata_test.obsm["X_scanvi"] = (
    scanvi.get_latent_representation(
        adata_test_subset
    )
)

print("latent shape:")
print(adata_test.obsm["X_scanvi"].shape)

INFO     File ../../resources/scvi_model_pert_50/model.pt already downloaded                                       
INFO     Found 100.0% reference vars in query data.                                                                


/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/model/base/_base_model.py:862: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


latent shape:
(4122, 50)


In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# flow block
# ---------------------------------------------------------

class FlowBlock(nn.Module):

    def __init__(
        self,
        latent_dim=50,
        cond_out_dim=64,
        time_dim=32,
        hidden=512,
        depth=2
    ):
        super().__init__()

        input_dim = latent_dim + time_dim + cond_out_dim

        layers = [
            nn.Linear(input_dim, hidden),
            nn.SiLU()
        ]

        for _ in range(depth):

            layers.append(
                nn.Linear(hidden, hidden)
            )

            layers.append(
                nn.SiLU()
            )

        layers.append(
            nn.Linear(hidden, latent_dim)
        )

        self.net = nn.Sequential(*layers)


    def forward(
        self,
        x,
        t_emb,
        cond_latent
    ):

        h = torch.cat(
            [
                x,
                t_emb,
                cond_latent
            ],
            dim=1
        )

        dx = self.net(h)

        return dx




# ---------------------------------------------------------
# flow model
# ---------------------------------------------------------

class FlowMLP(nn.Module):

    def __init__(
        self,
        latent_dim=50,
        cond_dim=1536,
        cond_out=64,
        n_flows=1,
        time_dim=32,
        cond_hidden = 512,
        hidden=512,
        depth=2,
        dropout = 0
    ):

        super().__init__()

        self.time_dim = time_dim


        # deeper condition encoder
        self.cond_net = nn.Sequential(

            nn.Linear(cond_dim, 512),
            nn.SiLU(),

            nn.Linear(512, 256),
            nn.SiLU(),

            nn.Linear(256, 128),
            nn.SiLU(),

            nn.Linear(128, cond_out),
            nn.SiLU()

        )


        self.blocks = nn.ModuleList(

            [

                FlowBlock(

                    latent_dim=latent_dim,

                    cond_out_dim=cond_out,

                    time_dim=time_dim,

                    hidden=hidden,

                    depth=depth

                )

                for _ in range(n_flows)

            ]

        )


    def forward(
        self,
        x,
        t,
        cond
    ):

        # -------------------------------------------------
        # normalize condition embedding
        # -------------------------------------------------

        cond = F.normalize(cond, dim=1)


        cond_latent = self.cond_net(cond)


        t_emb = fourier_time_embedding(

            t,

            dim=self.time_dim

        )


        step_scale = 1.0 / len(self.blocks)


        for block in self.blocks:

            dx = block(

                x,

                t_emb,

                cond_latent

            )

            x = x + step_scale * dx


        return x

In [ ]:
import math
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scripts.pairing import get_condition_encoder


# =========================================================
# 3. dataset
# =========================================================

class ConditionalFlowDataset(Dataset):
    """
    Online sampler for conditional flow matching.

    Each __getitem__ samples:
        condition c
        z1 ~ p_data(z | c)
        z0 ~ N(0, I)
    """

    def __init__(
        self,
        adata,
        encode_condition_fn,
        latent_key="X_scanvi",
        gene_col="gene",
        dataset_size=200_000,
        random_state=0,
        include_conditions=None
    ):
        self.adata = adata
        self.latent = np.asarray(adata.obsm[latent_key], dtype=np.float32)
        self.genes = adata.obs[gene_col].values
        self.encode_condition_fn = encode_condition_fn
        self.dataset_size = int(dataset_size)

        self.rng = np.random.default_rng(random_state)

        self.pert_to_idx = {}
        unique_perts = np.unique(self.genes)

        for pert in unique_perts:
            idx = np.where(self.genes == pert)[0]
            if len(idx) > 0:
                self.pert_to_idx[pert] = idx

        if include_conditions is not None:
            include_conditions = set(include_conditions)
            self.pert_to_idx = {
                k: v for k, v in self.pert_to_idx.items()
                if k in include_conditions
            }

        self.perts = sorted(list(self.pert_to_idx.keys()))

        if len(self.perts) == 0:
            raise ValueError("No perturbations found for dataset")

        self.latent_dim = self.latent.shape[1]

        # infer condition dimension from first perturbation
        test_cond = self.encode_condition_fn(self.perts[0])
        self.cond_dim = int(test_cond.shape[0])

    def __len__(self):
        return self.dataset_size

    def __getitem__(self, idx):
        pert = self.rng.choice(self.perts)

        z1_idx = self.rng.choice(self.pert_to_idx[pert])
        z1 = self.latent[z1_idx]

        z0 = self.rng.normal(
            loc=0.0,
            scale=1.0,
            size=self.latent_dim
        ).astype(np.float32)

        cond = self.encode_condition_fn(pert)

        return (
            torch.from_numpy(z0),
            torch.from_numpy(z1),
            torch.from_numpy(cond),
            pert
        )


# =========================================================
# 4. time embedding
# =========================================================

def fourier_time_embedding(
    t,
    dim=64,
    max_period=10000.0
):
    """
    Sinusoidal / Fourier time embedding.

    Parameters
    ----------
    t : torch.Tensor
        shape = (B, 1), values in [0, 1]
    dim : int
        embedding dimension
    max_period : float
        frequency scale

    Returns
    -------
    emb : torch.Tensor
        shape = (B, dim)
    """
    if t.ndim != 2 or t.shape[1] != 1:
        raise ValueError(f"t must have shape (B,1), got {tuple(t.shape)}")

    half_dim = dim // 2
    device = t.device
    dtype = t.dtype

    freqs = torch.exp(
        -math.log(max_period) * torch.arange(
            0, half_dim, device=device, dtype=dtype
        ) / max(half_dim - 1, 1)
    )

    # scale by 2pi so periodicity is more expressive
    args = 2.0 * math.pi * t * freqs.unsqueeze(0)

    emb = torch.cat(
        [torch.sin(args), torch.cos(args)],
        dim=1
    )

    if dim % 2 == 1:
        emb = torch.cat(
            [emb, torch.zeros((t.shape[0], 1), device=device, dtype=dtype)],
            dim=1
        )

    return emb


# =========================================================
# 5. model
# =========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F



# ------------------------------------------------------------
# Residual MLP block with time conditioning + dropout
# ------------------------------------------------------------

class FlowBlock(nn.Module):

    def __init__(
        self,
        latent_dim=50,
        cond_out_dim=32,
        time_dim=32,
        hidden=512,
        depth=2,
        dropout=0.1
    ):
        super().__init__()

        input_dim = latent_dim + cond_out_dim

        self.dropout = nn.Dropout(dropout)

        # input projection
        self.in_proj = nn.Linear(input_dim, hidden)

        # time projections (one per layer + input)
        self.time_proj = nn.ModuleList(
            [
                nn.Linear(time_dim, hidden)
                for _ in range(depth + 1)
            ]
        )

        # residual layers
        self.layers = nn.ModuleList(
            [
                nn.Sequential(
                    nn.LayerNorm(hidden),

                    nn.Linear(hidden, hidden),

                    nn.SiLU(),

                    nn.Dropout(dropout)

                )
                for _ in range(depth)
            ]
        )

        self.out_norm = nn.LayerNorm(hidden)

        self.out_proj = nn.Linear(hidden, latent_dim)


    def forward(
        self,
        x,
        t_emb,
        cond_latent
    ):

        # concatenate x and condition
        h = torch.cat(
            [
                x,
                cond_latent
            ],
            dim=1
        )


        # input projection
        h = self.in_proj(h)

        # inject time
        h = h + self.time_proj[0](t_emb)

        h = F.silu(h)

        h = self.dropout(h)


        # residual blocks
        for i, layer in enumerate(self.layers):

            residual = h

            h = layer(h)

            # inject time at each layer
            h = h + self.time_proj[i + 1](t_emb)

            # residual connection
            h = h + residual


        h = self.out_norm(h)

        dx = self.out_proj(h)

        return dx




# ------------------------------------------------------------
# Flow model
# ------------------------------------------------------------

class FlowMLP(nn.Module):

    def __init__(
        self,
        latent_dim=50,
        cond_dim=1536,
        cond_out=32,
        n_flows=20,
        time_dim=32,
        cond_hidden=256,
        hidden=512,
        depth=2,
        dropout=0.1
    ):

        super().__init__()

        self.time_dim = time_dim


        # condition encoder
        self.cond_net = nn.Sequential(

            nn.Linear(cond_dim, cond_hidden),

            nn.SiLU(),

            nn.Dropout(dropout),

            nn.Linear(cond_hidden, cond_out),

            nn.SiLU()

        )


        # flow blocks
        self.blocks = nn.ModuleList(

            [

                FlowBlock(

                    latent_dim=latent_dim,

                    cond_out_dim=cond_out,

                    time_dim=time_dim,

                    hidden=hidden,

                    depth=depth,

                    dropout=dropout

                )

                for _ in range(n_flows)

            ]

        )


    def forward(
        self,
        x,
        t,
        cond
    ):

        # encode condition once
        cond_latent = self.cond_net(cond)


        # Fourier time embedding
        t_emb = fourier_time_embedding(

            t,

            dim=self.time_dim

        )


        step_scale = 1.0 / len(self.blocks)


        # Euler ODE solver
        for block in self.blocks:

            dx = block(

                x,

                t_emb,

                cond_latent

            )

            x = x + step_scale * dx


        return x


# =========================================================
# 6. flow matching loss
# =========================================================

def flow_matching_loss(
    model,
    z0,
    z1,
    cond
):
    """
    Standard conditional flow matching loss with straight-line interpolation.

    z_t = (1 - t) z0 + t z1
    target velocity = z1 - z0
    """
    batch_size = z0.shape[0]
    device = z0.device

    t = torch.rand(batch_size, 1, device=device, dtype=z0.dtype)

    xt = (1.0 - t) * z0 + t * z1
    target = z1 - z0

    pred = model(xt, t, cond)

    loss = ((pred - target) ** 2).mean()
    return loss


# =========================================================
# 7. training
# =========================================================

def train_flow_model(
    adata,
    embedding_dict,
    cond_out_dim = 32,
    time_dim = 4,
    latent_key="X_scanvi",
    gene_col="gene",
    add_num_guides=True,
    normalize_embedding=False,
    batch_size=256,
    dataset_size=200_000,
    max_epochs=200,
    lr=1e-3,
    weight_decay=1e-4,
    device=None,
    random_state=0,
    model_save_path="best_flow_model.pt",
    num_workers=0
):
    if device is None:
        if torch.backends.mps.is_available():
            device = "mps"
        elif torch.cuda.is_available():
            device = "cuda"
        else:
            device = "cpu"

    encode_fn = get_condition_encoder(
        condition_mode="embedding",
        embedding_dict=embedding_dict,
        add_num_guides=add_num_guides,
    )
    all_perts = np.unique(adata.obs[gene_col])

    rng = np.random.default_rng(random_state)

    rng.shuffle(all_perts)

    split = int(len(all_perts) * 0.8)

    train_perts = all_perts[:split]
    val_perts   = all_perts[split:]
    
    val_dataset = ConditionalFlowDataset(
        adata=adata,
        encode_condition_fn=encode_fn,
        latent_key=latent_key,
        gene_col=gene_col,
        dataset_size=50_000,
        random_state=random_state + 1,
        include_conditions=val_perts
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=True
    )

    dataset = ConditionalFlowDataset(
        adata=adata,
        encode_condition_fn=encode_fn,
        latent_key=latent_key,
        gene_col=gene_col,
        dataset_size=dataset_size,
        random_state=random_state, 
        include_conditions=train_perts
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers
    )

    model = FlowMLP(
        latent_dim=dataset.latent_dim,
        cond_dim=dataset.cond_dim,
        time_dim=time_dim,
        cond_hidden=512,
        cond_out=cond_out_dim,
        hidden=512,
        depth=4,
        dropout=0.1,
        n_flows =1
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    history = []
    best_loss = float("inf")

    print(f"device: {device}")
    print(f"latent_dim: {dataset.latent_dim}")
    print(f"cond_dim: {dataset.cond_dim}")
    print(f"n_conditions: {len(dataset.perts)}")
    print("conditions:")
    print(dataset.perts)
    
    history = {
        "train": [],
        "val": []
    }

    best_loss = float("inf")

    for epoch in range(max_epochs):

        # =====================
        # train
        # =====================
        model.train()

        train_losses = []

        for z0, z1, cond, pert in loader:

            z0 = z0.to(device, dtype=torch.float32)
            z1 = z1.to(device, dtype=torch.float32)
            cond = cond.to(device, dtype=torch.float32)

            loss = flow_matching_loss(
                model=model,
                z0=z0,
                z1=z1,
                cond=cond
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())


        train_loss = float(np.mean(train_losses))


        # =====================
        # validation
        # =====================
        model.eval()

        val_losses = []

        with torch.no_grad():

            for z0, z1, cond, _ in val_loader:

                z0 = z0.to(device, dtype=torch.float32)
                z1 = z1.to(device, dtype=torch.float32)
                cond = cond.to(device, dtype=torch.float32)

                loss = flow_matching_loss(
                    model,
                    z0,
                    z1,
                    cond
                )

                val_losses.append(loss.item())


        val_loss = float(np.mean(val_losses))


        # =====================
        # record history
        # =====================
        history["train"].append(train_loss)
        history["val"].append(val_loss)


        # =====================
        # save best model (用 val)
        # =====================
        if val_loss < best_loss:

            best_loss = val_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),

                    "history": history,

                    "latent_dim": dataset.latent_dim,
                    "cond_dim": dataset.cond_dim,

                    "time_dim": time_dim,

                    "cond_out_dim": cond_out_dim,

                    "add_num_guides": add_num_guides,
                    "normalize_embedding": normalize_embedding,

                    "latent_key": latent_key,
                    "gene_col": gene_col,

                    "train_perts": dataset.perts,
                    "val_perts": val_dataset.perts,
                },
                model_save_path
            )


        # =====================
        # logging
        # =====================
        if epoch % 10 == 0 or epoch == max_epochs - 1:

            print(
                f"epoch {epoch:03d} | "
                f"train {train_loss:.6f} | "
                f"val {val_loss:.6f} | "
                f"best {best_loss:.6f}"
            )


    return model, history, dataset

# =========================================================
# 8. ODE sampling / inference
# =========================================================

@torch.no_grad()
def sample_latent_cells(
    model,
    condition,
    encode_condition_fn,
    latent_dim,
    n_cells=100,
    n_steps=100,
    device="cpu",
    z0=None
):
    """
    Generate latent samples for one condition using Euler ODE integration.

    dz/dt = v_theta(z, t, cond)
    """
    model.eval()

    if z0 is None:
        z = torch.randn(n_cells, latent_dim, device=device)
    else:
        z = torch.as_tensor(z0, dtype=torch.float32, device=device)
        if z.shape != (n_cells, latent_dim):
            raise ValueError(
                f"z0 shape must be {(n_cells, latent_dim)}, got {tuple(z.shape)}"
            )

    cond_vec = encode_condition_fn(condition)
    cond = torch.tensor(cond_vec, dtype=torch.float32, device=device).unsqueeze(0)
    cond = cond.repeat(n_cells, 1)

    dt = 1.0 / n_steps

    for step in range(n_steps):
        t_scalar = step / n_steps
        t = torch.full(
            (n_cells, 1),
            fill_value=t_scalar,
            device=device,
            dtype=torch.float32
        )

        v = model(z, t, cond)
        z = z + dt * v

    return z.detach().cpu().numpy()


# =========================================================
# 9. helper for batch generation
# =========================================================

@torch.no_grad()
def generate_for_conditions(
    model,
    conditions,
    encode_condition_fn,
    latent_dim,
    n_cells_per_condition=100,
    n_steps=100,
    device="cpu"
):
    """
    Generate latent samples for multiple perturbations.

    Returns
    -------
    generated_latents : np.ndarray
        shape = (sum cells, latent_dim)
    obs_df : pd.DataFrame
        contains gene column
    """
    all_latents = []
    all_obs = []

    for cond in conditions:
        z = sample_latent_cells(
            model=model,
            condition=cond,
            encode_condition_fn=encode_condition_fn,
            latent_dim=latent_dim,
            n_cells=n_cells_per_condition,
            n_steps=n_steps,
            device=device
        )

        all_latents.append(z)

        obs_df = pd.DataFrame({
            "gene": [cond] * n_cells_per_condition
        })
        all_obs.append(obs_df)

    generated_latents = np.vstack(all_latents).astype(np.float32)
    obs_df = pd.concat(all_obs, axis=0, ignore_index=True)

    return generated_latents, obs_df


In [60]:
import sys
sys.path.append('../')
from scripts.pairing import load_gene_embeddings


emb = load_gene_embeddings("large")
model, history, dataset = train_flow_model(
    adata=adata_train,
    embedding_dict=emb,
    cond_out_dim = 32,
    time_dim = 8,
    latent_key="X_scanvi",
    gene_col="gene",
    add_num_guides=False,
    normalize_embedding=False,
    batch_size=256,
    dataset_size=50000,
    max_epochs=200,
    lr=1e-4,
    weight_decay=1e-5,
    device="mps",   # or "cuda" / "cpu"
    random_state=0,
    model_save_path="try_best_flow_model.pt",
    num_workers=0
)

device: mps
latent_dim: 50
cond_dim: 3072
n_conditions: 162
conditions:
['CEBPA+CEBPA', 'CEBPA+CEBPA+CEBPB', 'CEBPA+CEBPA+NR3C1', 'CEBPA+CEBPA+POLR2D', 'CEBPA+CEBPA+STAT5B', 'CEBPA+CEBPB', 'CEBPA+CEBPB+KIF11', 'CEBPA+CEBPB+STAT5B', 'CEBPA+CEBPD', 'CEBPA+FOXO1', 'CEBPA+KIF11', 'CEBPA+KIF11+STAT5B', 'CEBPA+NC+NC', 'CEBPA+NC+STAT5B', 'CEBPA+NC+ZBED3', 'CEBPA+NR3C1', 'CEBPA+NR3C1+NR3C1', 'CEBPA+NR3C1+POLR2D', 'CEBPA+NR3C1+SF3B1', 'CEBPA+NR3C1+STAT5B', 'CEBPA+NR3C1+ZBED3', 'CEBPA+POLR2D', 'CEBPA+SF3B1', 'CEBPA+SF3B1+ZBED3', 'CEBPA+STAT5B+STAT5B', 'CEBPB+CEBPB', 'CEBPB+CEBPB+TCF7L2', 'CEBPB+CEBPD', 'CEBPB+KIF11', 'CEBPB+KIF11+KLF15', 'CEBPB+KIF11+NR3C1', 'CEBPB+KIF11+TCF7L2', 'CEBPB+KLF15+KLF15', 'CEBPB+KLF15+NC', 'CEBPB+KLF15+NR3C1', 'CEBPB+KLF15+POLR2D', 'CEBPB+KLF15+SF3B1', 'CEBPB+KLF15+STAT5B', 'CEBPB+NC+NR3C1', 'CEBPB+NC+POLR2D', 'CEBPB+NC+TCF7L2', 'CEBPB+NR3C1+POLR2D', 'CEBPB+NR3C1+STAT5B', 'CEBPB+POLR2D', 'CEBPB+SF3B1', 'CEBPD+KIF11', 'CEBPD+POLR2D', 'CEBPD+STAT5A', 'CEBPD+STAT5B', 'C

KeyboardInterrupt: 

In [45]:
import sys
sys.path.append('../')
from scripts.pairing import load_gene_embeddings, get_condition_encoder

emb = load_gene_embeddings("large")
ckpt = torch.load(
    "try_best_flow_model.pt",
    map_location="cpu"
)


model = FlowMLP(

    latent_dim = ckpt["latent_dim"],

    cond_dim = ckpt["cond_dim"],

    time_dim = ckpt["time_dim"],

    cond_hidden = 512,

    cond_out = ckpt["cond_out_dim"],

    hidden = 512,

    depth =3
)

model.load_state_dict(
    ckpt["model_state_dict"]
)


# ★ 這行是關鍵
device = "mps"

model = model.to(device)

model.eval()


encode_fn = get_condition_encoder(

    embedding_dict = emb,

    add_num_guides = ckpt["add_num_guides"],

    condition_mode = "embedding"
)


generated_latents, generated_obs = generate_for_conditions(

    model = model,

    conditions = val_condition,

    encode_condition_fn = encode_fn,

    latent_dim = ckpt["latent_dim"],

    n_cells_per_condition = 100,

    n_steps = 200,

    device = device
)

In [43]:
encode_fn = get_condition_encoder(
    embedding_dict=emb,
    add_num_guides=False,
    condition_mode='embedding'
)


generated_latents, generated_obs = generate_for_conditions(
    model=model,
    conditions=val_condition,
    encode_condition_fn=encode_fn,
    latent_dim=dataset.latent_dim,
    n_cells_per_condition=100,
    n_steps=400,
    device="mps"
)

print(generated_latents.shape)

NameError: name 'dataset' is not defined

In [46]:
import torch
import numpy as np


def decode_scanvi_latent(
    scanvi_model,
    latent,
    library_size=1e5,
    batch_size=512,
    device="cpu",
    mode="scale",
):

    device = torch.device(device)

    latent_t = torch.tensor(
        latent,
        dtype=torch.float32,
        device=device
    )

    decoder = scanvi_model.module.decoder
    dispersion = scanvi_model.module.dispersion

    outputs = []

    for i in range(0, latent_t.shape[0], batch_size):

        z = latent_t[i:i+batch_size]

        ##################################
        # library size
        ##################################

        library = torch.log(
            torch.ones(
                (z.shape[0], 1),
                device=device
            ) * library_size
        )

        ##################################
        # categorical covariates
        ##################################

        batch_index = torch.zeros(
            (z.shape[0], 1),
            dtype=torch.long,
            device=device
        )

        y = torch.zeros(
            (z.shape[0], 1),
            dtype=torch.long,
            device=device
        )

        ##################################
        # decode
        ##################################

        with torch.no_grad():

            px_scale, px_r, px_rate, px_dropout = decoder(

                dispersion,
                z,
                library,
                batch_index,
                y
            )

        ##################################
        # choose expression type
        ##################################

        if mode == "scale":

            expr = px_scale * library_size

        elif mode == "rate":

            expr = px_rate

        else:

            raise ValueError("mode must be scale or rate")

        outputs.append(

            expr.cpu().numpy()

        )

    return np.concatenate(outputs, axis=0)

In [47]:
gene_pred_decode = decode_scanvi_latent(
    scanvi,
    generated_latents,
    library_size=1e5,
    device="cpu",
)

gene_pred_decode

array([[ 1.2727901 ,  1.1120312 ,  7.3424244 , ...,  0.19196218,
         0.9750643 ,  0.01492692],
       [ 0.94685   ,  1.7645175 , 12.984014  , ...,  0.326473  ,
         1.184187  ,  0.0457561 ],
       [ 2.4086058 ,  3.1587243 ,  5.6003313 , ...,  0.13735513,
         1.4143734 ,  0.0328582 ],
       ...,
       [ 3.2996006 ,  6.693761  ,  7.712592  , ...,  0.21815713,
         2.1514795 ,  0.04134078],
       [ 2.389162  ,  4.5216303 ,  6.849609  , ...,  0.37847686,
         1.5821176 ,  0.08625873],
       [ 1.7809371 ,  1.1982901 ,  9.281785  , ...,  0.3440773 ,
         1.5844545 ,  0.01968415]], shape=(3400, 2546), dtype=float32)

In [48]:
from scripts.scoring import latent_to_expression_nn
gene_pred_nn = latent_to_expression_nn(

    latent_query = generated_latents,

    latent_train = adata_train.obsm["X_scanvi"],

    adata_train = adata_train,

    k = 1,

    method = "softmax" #distance,mean,softmax
)

In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

# 取出 PCA features
X_train_pca512 = adata_test.obsm["X_scanvi"]
X_test_pca512  = generated_latents

# 合併
X_all = np.vstack([X_train_pca512, X_test_pca512])

# 建立來源標籤
labels_source = (
    ["train"] * X_train_pca512.shape[0] +
    ["test"]  * X_test_pca512.shape[0]
)

labels_source = np.array(labels_source)

# 再做 PCA -> 2D 用於視覺化
pca_2d = PCA(n_components=2, random_state=42)

X_2d = pca_2d.fit_transform(X_all)

# 轉 dataframe
df_plot = pd.DataFrame({
    "PC1": X_2d[:,0],
    "PC2": X_2d[:,1],
    "dataset": labels_source
})

# 畫圖
plt.figure(figsize=(8,6))

for label in df_plot["dataset"].unique():

    subset = df_plot[df_plot["dataset"] == label]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=label,
        alpha=0.5,
        s=10
    )

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.title("PCA visualization (train vs test)")

plt.legend()

plt.show()

KeyError: 'X_scanvi'

In [49]:
import numpy as np


def merge_decoder_into_full_matrix(

    X_full,
    X_hvg,
    adata_full,
    scanvi_model
):
    """
    把 decoder 預測的 HVG expression 填回 full gene matrix

    Parameters
    ----------
    X_full
        (n_cells, n_all_genes)

    X_hvg
        (n_cells, n_hvg_genes)

    adata_full
        用來提供 gene name 順序

    scanvi_model
        用來取得 HVG gene list
    """

    ##################################
    # gene name list
    ##################################

    full_gene_names = adata_full.var_names.to_numpy()

    hvg_gene_names = scanvi_model.adata.var_names.to_numpy()

    ##################################
    # 找到 HVG 在 full matrix 中的位置
    ##################################

    gene_to_index = {

        g:i for i,g in enumerate(full_gene_names)

    }

    hvg_indices = [

        gene_to_index[g]

        for g in hvg_gene_names

        if g in gene_to_index

    ]

    ##################################
    # copy matrix
    ##################################

    X_merged = X_full.copy()

    ##################################
    # 覆蓋 HVG columns
    ##################################

    X_merged[:, hvg_indices] = X_hvg


    return X_merged

In [50]:
X_combined = merge_decoder_into_full_matrix(

    X_full = gene_pred_nn,

    X_hvg = gene_pred_decode,

    adata_full = adata_train,

    scanvi_model = scanvi
)

In [52]:
import importlib
import scripts.eval_flow

importlib.reload(scripts.eval_flow)

from scripts.eval_flow import evaluate_generated_expression_fast
results_df = evaluate_generated_expression_fast(

    pred_gex_matric = X_combined,

    generated_obs = generated_obs,

    conditions = val_condition,

    adata_gt_path = "../../data/obesity_challenge_2.h5ad",

    centroid_path = "../../data/preprocessed/centroid/centroid.csv",

)


Loading GT matrix...

Preparing gene sets...
{'random1000': 1000, 'hvg3000': 3000}
union genes: 3920

Preloading GT gene subset...

Evaluating CEBPD+CREB1

Evaluating KIF11+MLXIPL

Evaluating CEBPD+SREBF1

Evaluating CEBPD+ZBED3

Evaluating CEBPB+PPARG

Evaluating CEBPB+FOXO1

Evaluating CEBPA+MLXIPL

Evaluating STAT5A+TCF7L2

Evaluating MLXIPL+STAT5A

Evaluating CEBPB+NC

Evaluating CEBPD+PPARG

Evaluating PPARG+ZBED3

Evaluating PPARG+STAT5A

Evaluating NR3C1+PPARG2

Evaluating CREB1+STAT5A

Evaluating KLF15+PPARG

Evaluating PPARG2+SREBF1

Evaluating CEBPA+STAT5B

Evaluating CEBPA+CREB1

Evaluating MLXIPL+PPARG2

Evaluating PPARG2+TCF7L2

Evaluating KLF15+NC

Evaluating CEBPA+SREBF1

Evaluating CEBPA+PPARG2

Evaluating CEBPA+ZBED3

Evaluating CEBPB+TCF7L2

Evaluating NR3C1+PPARG

Evaluating PPARG2+ZBED3

Evaluating POLR2D+PPARG

Evaluating CEBPB+STAT5B

Evaluating STAT5A+ZBED3

Evaluating CEBPB+NR3C1

Evaluating SF3B1+SREBF1

Evaluating STAT5A+STAT5B

Summary
     perturbation  n_g

In [ ]:
pearson_random1000: 0.0588
mmd_random1000: 3.0496
pearson_hvg3000: 0.0448
mmd_hvg3000: 1.1285

In [ ]:
pearson_random1000: 0.0185
mmd_random1000: 0.0698
pearson_hvg3000: 0.0893
mmd_hvg3000: 0.1203